In [ ]:
import librosa
import numpy as np
import soundfile as sf
import os
from IPython.display import Audio, display
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from scipy.signal import resample_poly
from math import gcd

In [ ]:
# Load FLAC files
audio1, sr = librosa.load("../speaker_a.flac", sr=16000)
audio2, _ = librosa.load("../speaker_b.flac", sr=16000)

# Ensure equal length
length = min(len(audio1), len(audio2))


audio1 = audio1[:length]
audio2 = audio2[:length]

In [ ]:
print("Speaker A:", len(audio1) / sr / 60, "min")
print("Speaker B:", len(audio2) / sr / 60, "min")

In [ ]:
# Overlay both audio signals
mixed_audio = audio1 + audio2

In [ ]:
# Prevent clipping
scale = max(1, np.max(np.abs(mixed_audio)))
mixed_audio = mixed_audio / scale

# Check duration in minutes
print(len(mixed_audio) / sr / 60)

In [ ]:
os.makedirs("../full", exist_ok=True)

# clip_length = 180 * sr

# for i, start in enumerate(range(0, len(mixed_audio), clip_length)):

#     clip = mixed_audio[start:start + clip_length]

sf.write(f"../full/ab.flac", mixed_audio, sr)

# With all raw Data

In [ ]:
input_dir = Path("../raw_data/conversations")
output_dir = Path("../raw_data/fused")

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
conversations = sorted(input_dir.glob("conv_*"))

print("Total conversations:", len(conversations))

In [ ]:
conversations[:5]

In [ ]:
for conversation in tqdm(conversations):

    path_a = conversation / "speaker_a.flac"
    path_b = conversation / "speaker_b.flac"

    # Skip missing files
    if not path_a.exists() or not path_b.exists():
        print(f"Missing audio: {conversation.name}")
        continue

    # Define output filename
    output_path = output_dir / f"{conversation.name}.flac"

    with sf.SoundFile(path_a) as file_a, \
         sf.SoundFile(path_b) as file_b:

        # Check sample rates
        if file_a.samplerate != file_b.samplerate:
            print(f"Sample rate mismatch: {conversation.name}")
            continue

        sr = file_a.samplerate

        # Use shared audio length
        length = min(len(file_a), len(file_b))

        # Process 60 seconds at a time
        block_size = 60 * sr

        # Create fused audio file
        with sf.SoundFile(
            output_path,
            mode="w",
            samplerate=sr,
            channels=1,
            subtype="PCM_24"
        ) as output:

            for start in range(0, length, block_size):

                frames = min(block_size, length - start)

                # Read both speakers
                audio_a = file_a.read(
                    frames, dtype="float32", always_2d=True
                ).mean(axis=1)

                audio_b = file_b.read(
                    frames, dtype="float32", always_2d=True
                ).mean(axis=1)

                # Overlay both audio signals
                mixed_audio = (audio_a + audio_b) / 2

                # Write to the same output file
                output.write(mixed_audio)

In [ ]:
import pandas as pd
import re
from pathlib import Path

mapping_path = input_dir.parent / "name_mapping.csv"

# Load existing mapping
mapping = pd.read_csv(mapping_path, dtype=str)

print("Existing mappings:", len(mapping))

# Find all conversation folders
conversations = sorted(
    p for p in input_dir.glob("conv_*")
    if p.is_dir()
)

# Find the highest existing conversation number
used_numbers = [
    int(name.split("_")[-1])
    for name in mapping["conversation"]
]

used_numbers += [
    int(p.name.split("_")[-1])
    for p in conversations
    if re.fullmatch(r"conv_\d{1,6}", p.name)
]

next_number = max(used_numbers, default=0) + 1

# Update mapping with new conversations
for conversation in conversations:

    # Skip already renamed folders
    if conversation.name in mapping["conversation"].values:
        continue

    # Report numbered folders without a known mapping
    if re.fullmatch(r"conv_\d{1,6}", conversation.name):
        print(f"Unmapped numbered folder: {conversation.name}")
        continue

    # Check whether the original ID is already mapped
    existing = mapping[
        mapping["original_id"] == conversation.name
    ]

    if not existing.empty:
        new_name = existing.iloc[0]["conversation"]

    else:
        new_name = f"conv_{next_number}"
        fused_name = f"fused_audio{next_number}.flac"

        mapping.loc[len(mapping)] = {
            "original_id": conversation.name,
            "conversation": new_name,
            "fused_audio": fused_name
        }

        next_number += 1

    new_path = input_dir / new_name

    # Prevent overwriting existing folders
    if new_path.exists():
        raise FileExistsError(new_path)

    # Update CSV before renaming
    mapping.to_csv(mapping_path, index=False)

    # Rename conversation folder
    conversation.rename(new_path)

print("Updated mappings:", len(mapping))

# Create Dataframe

In [ ]:
records = []

for _, row in mapping.iterrows():

    conversation = input_dir / row["conversation"]

    fused = conversation / row["fused_audio"]
    audio_a = conversation / "speaker_a.flac"
    audio_b = conversation / "speaker_b.flac"

    # Only include complete conversations
    if all(p.exists() for p in [fused, audio_a, audio_b]):

        records.append({
            "conversation_id": row["conversation"],
            "original_id": row["original_id"],
            "fused": str(fused),
            "audio_a": str(audio_a),
            "audio_b": str(audio_b)
        })

df = pd.DataFrame(records)

df.head()

In [ ]:
X = df["fused"]

y = df[["audio_a", "audio_b"]]

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
df.to_csv(input_dir.parent / "dataset.csv", index=False)
# Saving our Datafraaaaaaaaamee